## 05 — FotMob Landing → Bronze

Reads FotMob JSON files from the Landing zone as binary files, parses them to VARIANT, and writes them as `bronze.fotmob_matches_raw`. FotMob is the second data source — it provides detailed shot data (xG, xGoT, position, shot type) and player statistics.


### 1. Configuration

`FOTMOB_PATH` points to JSON files pre-downloaded to the Unity Catalog Volume (FotMob has no public API — data is acquired separately). `FOTMOB_BRONZE_TABLE` is the fully-qualified target table name.


In [ ]:
from pyspark.sql import functions as F

FOTMOB_PATH = (
    "/Volumes/wsl_analytics/landing/fotmob_raw/"
    "match_details/"
    "league_id=9227/"
    "season=2025-2026/*.json"
)

FOTMOB_BRONZE_TABLE = (
    "wsl_analytics.bronze.fotmob_matches_raw"
)

### 2. Read as binary files 
Uses `binaryFile` format instead of standard `json` because we want full control over parsing — file content lands in a `content` column as bytes. Displays path, size, and modification time for each file.


In [ ]:
files_df = (
    spark.read
    .format("binaryFile")
    .load(FOTMOB_PATH)
)

display(
    files_df.select(
        "path",
        "length",
        "modificationTime"
    )
)

path,length,modificationTime
dbfs:/Volumes/wsl_analytics/landing/fotmob_raw/match_details/league_id=9227/season=2025-2026/match_4893195.json,943083,2026-08-14T19:41:26.000Z
dbfs:/Volumes/wsl_analytics/landing/fotmob_raw/match_details/league_id=9227/season=2025-2026/match_4892984.json,921638,2026-08-14T19:41:18.000Z
dbfs:/Volumes/wsl_analytics/landing/fotmob_raw/match_details/league_id=9227/season=2025-2026/match_4893204.json,900686,2026-08-14T19:41:27.000Z
dbfs:/Volumes/wsl_analytics/landing/fotmob_raw/match_details/league_id=9227/season=2025-2026/match_4893202.json,900534,2026-08-14T19:41:27.000Z
dbfs:/Volumes/wsl_analytics/landing/fotmob_raw/match_details/league_id=9227/season=2025-2026/match_4893185.json,888838,2026-08-14T19:41:25.000Z
dbfs:/Volumes/wsl_analytics/landing/fotmob_raw/match_details/league_id=9227/season=2025-2026/match_4893243.json,884845,2026-08-14T19:41:32.000Z
dbfs:/Volumes/wsl_analytics/landing/fotmob_raw/match_details/league_id=9227/season=2025-2026/match_4893144.json,879266,2026-08-14T19:41:24.000Z
dbfs:/Volumes/wsl_analytics/landing/fotmob_raw/match_details/league_id=9227/season=2025-2026/match_4892987.json,877200,2026-08-14T19:41:18.000Z
dbfs:/Volumes/wsl_analytics/landing/fotmob_raw/match_details/league_id=9227/season=2025-2026/match_4892980.json,874638,2026-08-14T19:41:18.000Z
dbfs:/Volumes/wsl_analytics/landing/fotmob_raw/match_details/league_id=9227/season=2025-2026/match_4893190.json,868571,2026-08-14T19:41:25.000Z


### 3. Count files

Verifies the FotMob file count — should match the number of season fixtures.


In [ ]:
print(
    "Number of FotMob files:",
    files_df.count()
)

Liczba plików FotMob: 132


### 4. Parse JSON → VARIANT 

Builds the bronze DataFrame:
- `regexp_extract` extracts the filename from the full path (last segment after `/`)
- `F.decode(content, "UTF-8")` converts bytes to string
- `try_parse_json()` parses the string to VARIANT — returns NULL for invalid JSON instead of raising
- `current_timestamp()` records the ingestion timestamp


In [ ]:
fotmob_bronze_df = (
    files_df
    .select(
        F.col("path")
            .alias("source_file"),

        F.regexp_extract(
            F.col("path"),
            r"([^/]+)$",
            1
        ).alias("source_file_name"),

        F.col("modificationTime")
            .alias("source_file_modification_time"),

        F.col("length")
            .alias("source_file_size_bytes"),

        F.try_parse_json(
            F.decode(
                F.col("content"),
                "UTF-8"
            )
        ).alias("payload"),

        F.current_timestamp()
            .alias("ingested_at")
    )
)

### 5. Schema 

Verifies `payload` column type is VARIANT.


In [ ]:
fotmob_bronze_df.printSchema()

root
 |-- source_file: string (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- source_file_modification_time: timestamp (nullable = true)
 |-- source_file_size_bytes: long (nullable = true)
 |-- payload: variant (nullable = true)
 |-- ingested_at: timestamp (nullable = false)



### 6. Extract key identifiers 

Extracts key fields from VARIANT into separate columns: `fotmob_match_id`, `league_id`, `home_team_id`, `away_team_id`, `match_datetime`, and `coverage_level`. Promoting these fields out of VARIANT enables fast filtering and joins without costly VARIANT parsing on every query.


In [ ]:
fotmob_bronze_df = (
    fotmob_bronze_df

    .withColumn(
        "fotmob_match_id",
        F.try_variant_get(
            "payload",
            "$.general.matchId",
            "bigint"
        )
    )

    .withColumn(
        "league_id",
        F.try_variant_get(
            "payload",
            "$.general.leagueId",
            "bigint"
        )
    )

    .withColumn(
        "home_team_id",
        F.try_variant_get(
            "payload",
            "$.general.homeTeam.id",
            "bigint"
        )
    )

    .withColumn(
        "away_team_id",
        F.try_variant_get(
            "payload",
            "$.general.awayTeam.id",
            "bigint"
        )
    )

    .withColumn(
        "match_datetime",
        F.to_timestamp(
            F.try_variant_get(
                "payload",
                "$.general.matchTimeUTCDate",
                "string"
            )
        )
    )

    .withColumn(
        "coverage_level",
        F.try_variant_get(
            "payload",
            "$.general.coverageLevel",
            "string"
        )
    )
)

### 7. Preview extracted IDs 

Displays match IDs, team IDs, and dates — extraction correctness check.


In [ ]:
display(
    fotmob_bronze_df.select(
        "fotmob_match_id",
        "league_id",
        "home_team_id",
        "away_team_id",
        "match_datetime",
        "coverage_level",
        "source_file_name"
    )
)

fotmob_match_id,league_id,home_team_id,away_team_id,match_datetime,coverage_level,source_file_name
4893195,9227,231488,258661,2026-02-01T14:30:00.000Z,xG,match_4893195.json
4892984,9227,231497,258661,2025-09-28T13:30:00.000Z,xG,match_4892984.json
4893204,9227,231494,628117,2026-02-15T12:00:00.000Z,xG,match_4893204.json
4893202,9227,231488,614954,2026-02-13T19:10:00.000Z,xG,match_4893202.json
4893185,9227,258661,258657,2026-01-24T12:30:00.000Z,xG,match_4893185.json
4893243,9227,231497,231488,2026-05-16T12:00:00.000Z,xG,match_4893243.json
4893144,9227,231488,231494,2025-12-14T11:55:00.000Z,xG,match_4893144.json
4892987,9227,231488,258657,2025-10-04T11:00:00.000Z,xG,match_4892987.json
4892980,9227,628117,231488,2025-09-19T18:30:00.000Z,xG,match_4892980.json
4893190,9227,258657,614954,2026-04-29T18:00:00.000Z,xG,match_4893190.json


### 8. Invalid JSON check 

Counts rows where `payload` is NULL — would indicate corrupted JSON files. Expected value is 0.


In [ ]:
invalid_json_count = (
    fotmob_bronze_df
    .filter(
        F.col("payload").isNull()
    )
    .count()
)

print(
    "Niepoprawne JSON-y:",
    invalid_json_count
)

Niepoprawne JSON-y: 0


### 9. Filename vs payload ID check 

Extracts the match ID from the filename (`match_12345.json`) and compares it with `fotmob_match_id` from the payload. Discrepancies would suggest incorrectly named or swapped files.


In [ ]:
fotmob_bronze_df = (
    fotmob_bronze_df
    .withColumn(
        "fotmob_match_id_filename",

        F.regexp_extract(
            F.col("source_file_name"),
            r"match_(\d+)\.json",
            1
        ).cast("long")
    )
)

### 10. Mismatch display 

Filters matches where filename ID != payload ID. Should return an empty result.


In [ ]:
display(
    fotmob_bronze_df
    .filter(
        F.col("fotmob_match_id")
        !=
        F.col("fotmob_match_id_filename")
    )
)

source_file,source_file_name,source_file_modification_time,source_file_size_bytes,payload,ingested_at,fotmob_match_id,league_id,home_team_id,away_team_id,match_datetime,coverage_level,fotmob_match_id_filename


### 11. Duplicate check 

Groups by `fotmob_match_id` and counts — each match should have exactly one file.


In [ ]:
duplicates_df = (
    fotmob_bronze_df
    .groupBy(
        "fotmob_match_id"
    )
    .count()
    .filter(
        F.col("count") > 1
    )
)

display(duplicates_df)

fotmob_match_id,count


### 12. League ID check 

Verifies all matches belong to the same league (WSL, ID 9227). A different league ID would indicate a data mix-up.


In [ ]:
(
    fotmob_bronze_df
    .groupBy(
        "league_id"
    )
    .count()
    .display()
)

league_id,count
9227,132


### 13. Write Bronze 

Writes to Delta Table. Full overwrite on each run.


In [ ]:
(
    fotmob_bronze_df
    .write
    .mode("overwrite")
    .saveAsTable(
        FOTMOB_BRONZE_TABLE
    )
)

### 14. Verify 

Displays and counts the written table.


In [ ]:
display(
    spark.table(
        "wsl_analytics.bronze.fotmob_matches_raw"
    )
)

source_file source_file_name source_file_modification_time source_file_size_bytes payload ingested_at fotmob_match_id league_id home_team_id away_team_id match_datetime coverage_level fotmob_match_id_filename dbfs:/Volumes/wsl_analytics/landing/fotmob_raw/match_details/league_id=9227/season=2025-2026/match_4893198.json match_4893198.json 2026-08-14T19:41:26.000Z 737004 {"content":{"buzz":null,"h2h":{"matches":[{"away":{"id":"258657","name":"Arsenal"},"finished":false,"home":{"id":"231488","name":"Manchester City"},"league":{"id":"9227","name":"WSL","pageUrl":"/leagues/9227/overview/wsl"},"matchUrl":"/matches/manchester-city-vs-arsenal/1j6la05u#4892987","status":{"awarded":false,"cancelled":false,"finished":true,"reason":{"long":"Full-Time","longKey":"finished","short":"FT","shortKey":"fulltime_short"},"scoreStr":"3 - 2","started":true,"utcTime":"2025-10-04T11:00:00.000Z"},"time":{"utcTime":"2025-10-04T11:00:00.000Z"}},{"away":{"id":"231488","name":"Manchester City"},"finished":false,"home":{"id":"258657","name":"Arsenal"},"league":{"id":"9717","name":"Women's League Cup Final Stage","pageUrl":"/leagues/9717/overview/womens-league-cup-final-stage"},"matchUrl":"/matches/manchester-city-vs-arsenal/1j6la05u#4719588","status":{"awarded":false,"cancelled":false,"finished":true,"reason":{"long":"Full-Time","longKey":"finished","short":"FT","shortKey":"fulltime_short"},"scoreStr":"1 - 2","started":true,"utcTime":"2025-02-06T19:30:00.000Z"},"time":{"utcTime":"2025-02-06T19:30:00.000Z"}},{"away":{"id":"258657","name":"Arsenal"},"finished":false,"home":{"id":"231488","name":"Manchester City"},"league":{"id":"9227","name":"WSL","pageUrl":"/leagues/9227/overview/wsl"},"matchUrl":"/matches/manchester-city-vs-arsenal/1j6la05u#4570568","status":{"awarded":false,"cancelled":false,"finished":true,"reason":{"long":"Full-Time","longKey":"finished","short":"FT","shortKey":"fulltime_short"},"scoreStr":"3 - 4","started":true,"utcTime":"2025-02-02T12:00:00.000Z"},"time":{"utcTime":"2025-02-02T12:00:00.000Z"}},{"away":{"id":"231488","name":"Manchester City"},"finished":false,"home":{"id":"258657","name":"Arsenal"},"league":{"id":"9227","name":"WSL","pageUrl":"/leagues/9227/overview/wsl"},"matchUrl":"/matches/manchester-city-vs-arsenal/1j6la05u#4570479","status":{"awarded":false,"cancelled":false,"finished":true,"reason":{"long":"Full-Time","longKey":"finished","short":"FT","shortKey":"fulltime_short"},"scoreStr":"2 - 2","started":true,"utcTime":"2024-09-22T11:30:00.000Z"},"time":{"utcTime":"2024-09-22T11:30:00.000Z"}},{"away":{"id":"258657","name":"Arsenal"},"finished":false,"home":{"id":"231488","name":"Manchester City"},"league":{"id":"9227","name":"WSL","pageUrl":"/leagues/9227/overview/wsl"},"matchUrl":"/matches/manchester-city-vs-arsenal/1j6la05u#4255842","status":{"awarded":false,"cancelled":false,"finished":true,"reason":{"long":"Full-Time","longKey":"finished","short":"FT","shortKey":"fulltime_short"},"scoreStr":"1 - 2","started":true,"utcTime":"2024-05-05T13:15:00.000Z"},"time":{"utcTime":"2024-05-05T13:15:00.000Z"}},{"away":{"id":"231488","name":"Manchester City"},"finished":false,"home":{"id":"258657","name":"Arsenal"},"league":{"id":"10082","name":"Women's FA Cup","pageUrl":"/leagues/10082/overview/womens-fa-cup"},"matchUrl":"/matches/manchester-city-vs-arsenal/1j6la05u#4402030","status":{"awarded":false,"cancelled":false,"finished":true,"reason":{"long":"Full-Time","longKey":"finished","short":"FT","shortKey":"fulltime_short"},"scoreStr":"0 - 1","started":true,"utcTime":"2024-02-11T12:30:00.000Z"},"time":{"utcTime":"2024-02-11T12:30:00.000Z"}},{"away":{"id":"231488","name":"Manchester City"},"finished":false,"home":{"id":"258657","name":"Arsenal"},"league":{"id":"9227","name":"WSL","pageUrl":"/leagues/9227/overview/wsl"},"matchUrl":"/matches/manchester-city-vs-arsenal/1j6la05u#4255775","status":{"awarded":false,"cancelled":false,"finished":true,"reason":{"long":"Full-Time","longKey":"finished","short":"FT","shortKey":"fulltime_short"},"sc

In [ ]:
print(
    "Bronze FotMob:",
    spark.table(
        "wsl_analytics.bronze.fotmob_matches_raw"
    ).count()
)

Bronze FotMob: 132
